# Global Linear RRF Weights With LLM Judge

This notebook implements the exact global-weight experiment:

1. Split UC1 and UC2 into training and validation queries.
2. Run BM25 + cross-encoder candidate ranking and RRF.
3. Let the LLM judge improve the four query variants.
4. Run BM25 + cross-encoder candidate ranking again and RRF.
5. Use the resulting rankings to learn one global set of RRF weights.
6. Apply the full judged pipeline to test data: the UC1/UC2 validation halves and all UC3 queries.
7. Evaluate recall, precision, and accuracy.

Only one weight vector is learned across UC1 and UC2, so the model is more generalizable than per-use-case or per-level weights.

## Method

For each original query $q$, the pipeline uses five retrieval inputs:

$$
V = \{v_0, v_1, v_2, v_3, v_4\}
$$

where $v_0$ is the original query and $v_1,\dots,v_4$ are the legal terminology, compliance, contract, and risk variants.

For each variant $v_i$, BM25 retrieves candidates and the cross-encoder reranks those candidates with the original query $q$. The rank-based feature is:

$$
x_i(q,d) =
\begin{cases}
\frac{1}{K + r_i(q,d)} & \text{if document } d \text{ is retrieved by variant } v_i \\
0 & \text{otherwise}
\end{cases}
$$

A single global logistic regression is trained over all UC1/UC2 training examples:

$$
P(y=1 \mid q,d) = \sigma\left(\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}(q,d)\right)
$$

The final RRF weights are obtained by clipping negative coefficients and normalizing:

$$
w_i = \frac{\max(\beta_i, 0)}{\sum_j \max(\beta_j, 0)}
$$

Final ranking uses:

$$
s(q,d) = \sum_i w_i x_i(q,d)
$$

In [2]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import main
importlib.reload(main)

from main import JudgeContext, QueryVariants, clean_text, llm_judge_refine_queries
from retrieval.retrieval_bm25 import Query, build_bm25_index, load_corpus
from retrieval.sota_retrieval import SotaRetriever

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Configuration

`RUN_LLM_JUDGE` controls whether missing judged variants are generated. Judged variants are cached in `output_with_agents_<use_case>.csv` under `global_judge_*` columns.

In [3]:
TRAIN_USE_CASES = ["uc1", "uc2"]
EXTERNAL_TEST_USE_CASES = ["uc3"]
LEVELS = ["process", "subprocess", "task"]

RRF_K = 60
RANDOM_STATE = 42
JUDGE_PROVIDER = "gemini"
RUN_LLM_JUDGE = True

BASE_VARIANTS = [
    ("baseline", "query"),
    ("legal_terminology_rewrite", "legal_terminology_rewrite"),
    ("regulatory_compliance_query", "regulatory_compliance_query"),
    ("contract_clause_query", "contract_clause_query"),
    ("risk_scenario_query", "risk_scenario_query"),
]

GLOBAL_JUDGE_VARIANTS = [
    ("baseline", "query"),
    ("global_judge_legal_terminology_rewrite", "global_judge_legal_terminology_rewrite"),
    ("global_judge_regulatory_compliance_query", "global_judge_regulatory_compliance_query"),
    ("global_judge_contract_clause_query", "global_judge_contract_clause_query"),
    ("global_judge_risk_scenario_query", "global_judge_risk_scenario_query"),
]

LEVEL_TO_GS_SUFFIX = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

LEVEL_TO_TOP_K = {
    "process": 100,
    "subprocess": 30,
    "task": 15,
}

INITIAL_RRF_WEIGHTS = np.ones(len(BASE_VARIANTS)) / len(BASE_VARIANTS)

In [1]:
def corpus_path_for(use_case):
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx")


def gold_path_for(use_case, level):
    suffix = LEVEL_TO_GS_SUFFIX[level]
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx")


def recorded_path_for(use_case):
    return Path(f"output_with_agents_{use_case}.csv")


def load_records(use_case):
    path = recorded_path_for(use_case)
    if not path.exists():
        raise FileNotFoundError(f"Missing recorded query variants: {path}")
    df = pd.read_csv(path)
    df["level"] = df["level"].astype(str).str.lower()
    df["query_clean"] = df["query"].apply(clean_text)
    return df


def save_records(use_case, records_df):
    path = recorded_path_for(use_case)
    records_df.drop(columns=["query_clean"], errors="ignore").to_csv(path, index=False)
    print(f"Saved {path}")


def variant_text(row, column):
    if column not in row.index:
        return clean_text(row["query"])
    value = row[column]
    if value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == "":
        return clean_text(row["query"])
    return clean_text(value)


def split_queries(queries):
    queries = list(queries)
    if len(queries) == 1:
        return queries, queries
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(queries))
    rng.shuffle(indices)
    split_at = max(1, len(indices) // 2)
    train_indices = set(indices[:split_at])
    train_queries = [query for idx, query in enumerate(queries) if idx in train_indices]
    validation_queries = [query for idx, query in enumerate(queries) if idx not in train_indices]
    return train_queries, validation_queries

## BM25 + CE + RRF Helpers

Each variant retrieves BM25 candidates. The cross-encoder reranks those candidates with the original query. RRF then combines the variant lists.

In [4]:
def ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k):
    original_query = clean_text(row["query"])
    ranked_lists = {}
    for variant_name, column in variant_columns:
        bm25_results = bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k)
        if not bm25_results:
            ranked_lists[variant_name] = []
            continue
        cross_inputs = [[original_query, documents[result.document.doc_id].text] for result in bm25_results]
        cross_scores = cross_encoder.predict(cross_inputs)
        scored = [
            (result.document.doc_id, float(score))
            for result, score in zip(bm25_results, cross_scores, strict=True)
        ]
        scored.sort(key=lambda item: item[1], reverse=True)
        ranked_lists[variant_name] = [
            {"doc_id": doc_id, "score": score, "rank": rank}
            for rank, (doc_id, score) in enumerate(scored, start=1)
        ]
    return ranked_lists


def rrf_feature_rows(ranked_lists, variant_columns):
    doc_features = {}
    for variant_index, (variant_name, _) in enumerate(variant_columns):
        for item in ranked_lists.get(variant_name, []):
            features = doc_features.setdefault(item["doc_id"], np.zeros(len(variant_columns), dtype=float))
            features[variant_index] = 1.0 / (RRF_K + item["rank"])
    return doc_features


def rank_with_rrf(row, variant_columns, weights, bm25_index, documents, cross_encoder, top_k):
    ranked_lists = ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k)
    doc_features = rrf_feature_rows(ranked_lists, variant_columns)
    scored_docs = [(doc_id, float(np.dot(features, weights))) for doc_id, features in doc_features.items()]
    scored_docs.sort(key=lambda item: item[1], reverse=True)
    return scored_docs[:top_k], ranked_lists, doc_features


def ranked_docs_to_context_df(scored_docs, documents, query, level):
    return pd.DataFrame(
        [
            {
                "level": level,
                "query": query,
                "rel_text": clean_text(documents[doc_id].text),
                "score": score,
                "method": "rrf_ce_context",
                "query_variant": "rrf_ce_context",
                "source_variants": "rrf_ce_context",
            }
            for doc_id, score in scored_docs
        ]
    )

## LLM Judge Query Improvement

The judge receives the top 15 original BM25 results and top 15 initial RRF+CE results, then rewrites the four non-baseline variants.

In [5]:
def base_variants_from_row(row):
    return QueryVariants(
        legal_terminology_rewrite=variant_text(row, "legal_terminology_rewrite"),
        regulatory_compliance_query=variant_text(row, "regulatory_compliance_query"),
        contract_clause_query=variant_text(row, "contract_clause_query"),
        risk_scenario_query=variant_text(row, "risk_scenario_query"),
    )


def judge_columns_missing(row):
    required = [column for _, column in GLOBAL_JUDGE_VARIANTS if column != "query"]
    return any(column not in row.index or pd.isna(row[column]) or str(row[column]).strip() == "" for column in required)


def improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k):
    if not RUN_LLM_JUDGE:
        raise ValueError("Missing global judge variants. Set RUN_LLM_JUDGE = True to generate them.")

    initial_ranked, _, _ = rank_with_rrf(
        row=row,
        variant_columns=BASE_VARIANTS,
        weights=INITIAL_RRF_WEIGHTS,
        bm25_index=bm25_index,
        documents=documents,
        cross_encoder=sota_retriever.cross_encoder,
        top_k=top_k,
    )
    baseline_bm25 = bm25_index.rank(Query(text=clean_text(row["query"])), top_k=top_k)
    ce_context = ranked_docs_to_context_df(initial_ranked[:15], documents, clean_text(row["query"]), level)
    return llm_judge_refine_queries(
        JudgeContext(
            original_query=clean_text(row["query"]),
            variants=base_variants_from_row(row),
            bm25_results=baseline_bm25[:15],
            ce_rows=ce_context,
        ),
        documents=documents,
        judge_provider=JUDGE_PROVIDER,
    )


def ensure_global_judge_variants(use_case, records_df):
    required = [column for _, column in GLOBAL_JUDGE_VARIANTS if column != "query"]
    for column in required:
        if column not in records_df.columns:
            records_df[column] = ""

    missing_mask = records_df.apply(judge_columns_missing, axis=1)
    if not missing_mask.any():
        print(f"{use_case}: global judge variants already available.")
        return records_df

    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])

    for row_index, row in records_df[missing_mask].iterrows():
        level = row["level"]
        top_k = LEVEL_TO_TOP_K[level]
        print(f"[{use_case} / {level}] LLM judge query improvement for row {row_index}")
        judged = improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k)
        records_df.loc[row_index, "global_judge_legal_terminology_rewrite"] = judged.legal_terminology_rewrite
        records_df.loc[row_index, "global_judge_regulatory_compliance_query"] = judged.regulatory_compliance_query
        records_df.loc[row_index, "global_judge_contract_clause_query"] = judged.contract_clause_query
        records_df.loc[row_index, "global_judge_risk_scenario_query"] = judged.risk_scenario_query

    save_records(use_case, records_df)
    return records_df

## Split UC1/UC2 And Learn One Global Weight Vector

The training examples come from all UC1/UC2 training queries across all levels. This produces one shared vector $\mathbf{w}$.

In [7]:
def build_gold_lookup(use_case, level, documents):
    gold_df = pd.read_excel(gold_path_for(use_case, level))
    gold_df["query_clean"] = gold_df["query"].apply(clean_text)
    doc_id_by_text = {clean_text(document.text): document.doc_id for document in documents}

    gold_lookup = {}
    for query, group in gold_df.groupby("query_clean"):
        gold_doc_ids = {
            doc_id_by_text[clean_text(rel_text)]
            for rel_text in group["rel_text"].tolist()
            if clean_text(rel_text) in doc_id_by_text
        }
        gold_lookup[query] = gold_doc_ids
    return gold_lookup


def level_queries(records_df, gold_lookup, level):
    level_records = records_df[records_df["level"] == level]
    return [query for query in level_records["query_clean"].tolist() if query in gold_lookup]


def build_examples_for_queries(use_case, level, records_df, query_subset):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()

    examples = []
    labels = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        _, _, doc_features = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=INITIAL_RRF_WEIGHTS,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        positives = gold_lookup.get(query, set())
        for doc_id, features in doc_features.items():
            examples.append(features)
            labels.append(1 if doc_id in positives else 0)
    return examples, labels


records_by_use_case = {}
splits = []
all_examples = []
all_labels = []

for use_case in TRAIN_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        train_queries, validation_queries = split_queries(queries)
        splits.append(
            {
                "use_case": use_case,
                "level": level,
                "train_queries": train_queries,
                "validation_queries": validation_queries,
            }
        )
        examples, labels = build_examples_for_queries(use_case, level, records_df, train_queries)
        all_examples.extend(examples)
        all_labels.extend(labels)

x_train = np.array(all_examples)
y_train = np.array(all_labels)
print(f"Training examples: {len(x_train)}")
print(f"Positive labels: {int(y_train.sum())}")

uc1: global judge variants already available.
uc2: global judge variants already available.
Training examples: 1817
Positive labels: 125


In [8]:
def learn_global_weights(x_train, y_train):
    if len(x_train) == 0 or len(np.unique(y_train)) < 2:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"

    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(x_train, y_train)
    raw_weights = np.maximum(model.coef_[0], 0.0)
    if raw_weights.sum() == 0:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"
    return raw_weights / raw_weights.sum(), "global_linear_logistic_coefficients"


global_weights, weight_source = learn_global_weights(x_train, y_train)
weights_df = pd.DataFrame(
    [
        {
            "weight_source": weight_source,
            "training_use_cases": ",".join(TRAIN_USE_CASES),
            "training_examples": len(x_train),
            "positive_labels": int(y_train.sum()),
            **{variant_name: weight for (variant_name, _), weight in zip(GLOBAL_JUDGE_VARIANTS, global_weights)},
        }
    ]
)
weights_df

,weight_source,training_use_cases,training_examples,positive_labels,baseline,global_judge_legal_terminology_rewrite,global_judge_regulatory_compliance_query,global_judge_contract_clause_query,global_judge_risk_scenario_query
0,global_linear_logistic_coefficients,"uc1,uc2",1817,125,0.178309,0.223762,0.246037,0.168128,0.183764


## Apply Pipeline To Test Data

Test data consists of:

- UC1 validation split
- UC2 validation split
- all UC3 queries

For each test query, the notebook uses the judged variants, reruns BM25 + CE for each variant, merges with global weighted RRF, and evaluates the final ranking.

In [9]:
def evaluate_prediction_sets(predictions, gold_lookup, corpus_size):
    tp = fp = fn = tn = 0
    for query, predicted in predictions.items():
        gold = gold_lookup.get(query, set())
        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)
        tn += corpus_size - len(gold | predicted)

    accuracy = (tp + tn) / (tp + fp + fn + tn) if tp + fp + fn + tn else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    return {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }


def apply_global_pipeline(use_case, level, records_df, query_subset, split_name):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()

    predictions = {}
    rows = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        ranked_docs, _, _ = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=global_weights,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        predictions[query] = {doc_id for doc_id, _ in ranked_docs}
        for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            rows.append(
                {
                    "use_case": use_case,
                    "level": level,
                    "split": split_name,
                    "query": query,
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": score,
                    "rel_text": clean_text(documents[doc_id].text),
                }
            )

    metrics = evaluate_prediction_sets(predictions, gold_lookup, len(documents))
    metrics.update(
        {
            "use_case": use_case,
            "level": level,
            "split": split_name,
            "queries": len(query_subset),
        }
    )
    return metrics, rows


metric_rows = []
prediction_rows = []

for split in splits:
    metrics, rows = apply_global_pipeline(
        use_case=split["use_case"],
        level=split["level"],
        records_df=records_by_use_case[split["use_case"]],
        query_subset=split["validation_queries"],
        split_name="validation",
    )
    metric_rows.append(metrics)
    prediction_rows.extend(rows)

for use_case in EXTERNAL_TEST_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        metrics, rows = apply_global_pipeline(
            use_case=use_case,
            level=level,
            records_df=records_df,
            query_subset=queries,
            split_name="external_test",
        )
        metric_rows.append(metrics)
        prediction_rows.extend(rows)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
metrics_df

uc3: global judge variants already available.


,true_positives,false_positives,false_negatives,true_negatives,accuracy,precision,recall,use_case,level,split,queries
0,16,84,33,356,0.760736,0.160000,0.326531,uc1,process,validation,1
1,10,110,28,1808,0.929448,0.083333,0.263158,uc1,subprocess,validation,4
2,19,206,54,7056,0.964554,0.084444,0.260274,uc1,task,validation,15
3,27,73,4,207,0.752412,0.270000,0.870968,uc2,process,validation,1
4,22,98,18,1106,0.906752,0.183333,0.550000,uc2,subprocess,validation,4
5,27,123,38,2922,0.948232,0.180000,0.415385,uc2,task,validation,10
6,17,83,0,68,0.505952,0.170000,1.000000,uc3,process,external_test,1
7,32,118,15,675,0.841667,0.213333,0.680851,uc3,subprocess,external_test,5
8,44,256,39,3021,0.912202,0.146667,0.530120,uc3,task,external_test,20


## Summary Table

This table reports recall, precision, and accuracy for each use case and level.

In [10]:
summary_df = metrics_df[
    [
        "use_case",
        "level",
        "split",
        "queries",
        "true_positives",
        "false_positives",
        "false_negatives",
        "true_negatives",
        "recall",
        "precision",
        "accuracy",
    ]
].copy()
summary_df[["recall", "precision", "accuracy"]] = summary_df[["recall", "precision", "accuracy"]].round(3)
summary_df

,use_case,level,split,queries,true_positives,false_positives,false_negatives,true_negatives,recall,precision,accuracy
0,uc1,process,validation,1,16,84,33,356,0.327,0.160,0.761
1,uc1,subprocess,validation,4,10,110,28,1808,0.263,0.083,0.929
2,uc1,task,validation,15,19,206,54,7056,0.260,0.084,0.965
3,uc2,process,validation,1,27,73,4,207,0.871,0.270,0.752
4,uc2,subprocess,validation,4,22,98,18,1106,0.550,0.183,0.907
5,uc2,task,validation,10,27,123,38,2922,0.415,0.180,0.948
6,uc3,process,external_test,1,17,83,0,68,1.000,0.170,0.506
7,uc3,subprocess,external_test,5,32,118,15,675,0.681,0.213,0.842
8,uc3,task,external_test,20,44,256,39,3021,0.530,0.147,0.912


## Export Results

The workbook stores the single global weight vector, split definitions, evaluation metrics, and ranked predictions.

In [11]:
export_path = project_root / "linear_rrf_global_weights_llm_judge_results.xlsx"

split_rows = []
for split in splits:
    split_rows.append(
        {
            "use_case": split["use_case"],
            "level": split["level"],
            "train_queries": len(split["train_queries"]),
            "validation_queries": len(split["validation_queries"]),
        }
    )
splits_df = pd.DataFrame(split_rows)

with pd.ExcelWriter(export_path) as writer:
    weights_df.to_excel(writer, sheet_name="global_weights", index=False)
    splits_df.to_excel(writer, sheet_name="splits", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    predictions_df.to_excel(writer, sheet_name="ranked_predictions", index=False)

print(f"Exported {export_path}")

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/linear_rrf_global_weights_llm_judge_results.xlsx
